# IPG Selection Emulator

Run the selection logic from `parse_ipg.py` on a raw IPG TSV (e.g., `/tmp/apl18794_ipg.txt`).

In [1]:
import pandas as pd
import polars as pl
from pathlib import Path
import sys

sys.path.append('src')
from hoodini.pipeline.parse_ipg import _select_best_ipg

IPG_TXT = Path('/tmp/apl18794_ipg.txt')
QUERY_PROTEIN = 'APL18794.1'
CAND_MODE = 'best_ipg'

if not IPG_TXT.exists():
    raise FileNotFoundError(f'IPG TXT not found at {IPG_TXT}')

raw_pd = pd.read_csv(IPG_TXT, sep='	', dtype=str)
ipg_df = pl.from_pandas(raw_pd)

ipg_df = ipg_df.rename({
    'Source': 'source',
    'Nucleotide Accession': 'nucleotide_id',
    'Start': 'start',
    'Stop': 'end',
    'Strand': 'strand',
    'Protein': 'protein_id',
    'Protein Name': 'protein name',
    'Assembly': 'assembly_id',
    'Id': 'ipg_id',
})

ipg_df = ipg_df.with_columns([
    pl.lit(0).alias('og_index'),
    pl.lit(QUERY_PROTEIN).alias('query_protein_id'),
    pl.col('protein_id').str.starts_with(('WP_', 'YP_')).alias('is_refseq_query'),
])

ipg_df.head()


InvalidOperationError: cannot cast List type (inner: 'String', to: 'String')

In [ ]:
selected = _select_best_ipg(ipg_df, cand_mode=CAND_MODE)
selected
